# FinGuard AI — Final Modeling & Explainability

This notebook presents the final XGBoost evaluation and model interpretation. Data audit, target construction, feature screening, and leakage checks are handled in the preceding notebook.

**Validation design:** train on FY2012–2019, validate on FY2020–2021, and evaluate out-of-time on FY2022–2023. ICR is excluded from the predictive feature set because it contributes to the distress-label construction.

In [ ]:
# FINAL OOT EVALUATION + PROJECT CONCLUSION

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    log_loss,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    brier_score_loss
)

from xgboost import XGBClassifier

print("=" * 80)
print("FINANCIAL DISTRESS PROJECT — STEP 10")
print("FINAL OOT EVALUATION")
print("=" * 80)

# 1. LOAD DATA

data_path = r"./data/distress_modeling_base.csv"

df = pd.read_csv(data_path)

df = df.replace([np.inf, -np.inf], np.nan)

print("\nDATASET")
print("-" * 80)
print("Shape:", df.shape)

# 2. TARGET ENCODING

target_map = {
    "Healthy": 0,
    "Vulnerable": 1,
    "Distressed": 2
}

df["Target"] = df["DistressLabel"].map(target_map)

# 3. FEATURE LIST

feature_cols = [
    "Leverage_trend",
    "SalesGrowth_3yr_trend",
    "InstitutionalHolding_pct",
    "Industry_Median_Deviation",
    "DebtEquityRatio",
    "CashConversionCycle",
    "InventoryDays",
    "CreditorDays",
    "ROCE_3yr_trend",
    "SalesGrowth",
    "PromoterControlConcentration",
    "Accruals_to_Assets",
    "CFO_to_NetIncome",
    "CFO_to_TL",
    "DebtorDays",
    "CashBurnRunway",
    "TangibleAssetRatio",
    "AssetTurnover",
    "CashToAssets",
    "CurrentRatio",
    "FirmAge",
    "ROCE",
    "LogTotalAssets",
    "EquityRaised_t",
    "PromoterPledgeRatio",
    "PledgeDisclosed",
    "EquityRaised_dummy"
]

print("\nFEATURES")
print("-" * 80)
print("Number of features:", len(feature_cols))

# 4. TEMPORAL SPLIT

train = df[df["FY"].between(2012, 2019)].copy()
validation = df[df["FY"].between(2020, 2021)].copy()
oot = df[df["FY"].between(2022, 2023)].copy()

X_train = train[feature_cols]
y_train = train["Target"]

X_validation = validation[feature_cols]
y_validation = validation["Target"]

X_oot = oot[feature_cols]
y_oot = oot["Target"]

print("\nTEMPORAL SPLIT")
print("-" * 80)
print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("OOT:", X_oot.shape)

# 5. TRAIN FINAL XGBOOST

print("\n" + "=" * 80)
print("TRAINING FINAL XGBOOST")
print("=" * 80)

final_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

final_xgb.fit(X_train, y_train)

print("✓ Final XGBoost fitted.")

# 6. OOT PREDICTIONS

oot_prob = final_xgb.predict_proba(X_oot)
oot_pred = np.argmax(oot_prob, axis=1)

print("✓ OOT predictions generated.")

# 7. FINAL OOT METRICS

accuracy = accuracy_score(y_oot, oot_pred)

balanced_acc = balanced_accuracy_score(
    y_oot,
    oot_pred
)

macro_f1 = f1_score(
    y_oot,
    oot_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_oot,
    oot_pred,
    average="weighted"
)

macro_precision = precision_score(
    y_oot,
    oot_pred,
    average="macro"
)

macro_recall = recall_score(
    y_oot,
    oot_pred,
    average="macro"
)

logloss = log_loss(
    y_oot,
    oot_prob
)

roc_auc = roc_auc_score(
    y_oot,
    oot_prob,
    multi_class="ovr",
    average="macro"
)

print("\n" + "=" * 80)
print("FINAL OOT PERFORMANCE — 2022–2023")
print("=" * 80)

final_metrics = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Log Loss",
        "ROC-AUC OVR"
    ],
    "Value": [
        accuracy,
        balanced_acc,
        macro_precision,
        macro_recall,
        macro_f1,
        weighted_f1,
        logloss,
        roc_auc
    ]
})

print(final_metrics.to_string(index=False))

# 8. CLASSIFICATION REPORT

print("\n" + "=" * 80)
print("FINAL OOT CLASSIFICATION REPORT")
print("=" * 80)

class_names = [
    "Healthy",
    "Vulnerable",
    "Distressed"
]

report_dict = classification_report(
    y_oot,
    oot_pred,
    target_names=class_names,
    output_dict=True
)

report_df = pd.DataFrame(report_dict).T

print(
    classification_report(
        y_oot,
        oot_pred,
        target_names=class_names
    )
)

# 9. CONFUSION MATRIX

cm = confusion_matrix(
    y_oot,
    oot_pred
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual Healthy",
        "Actual Vulnerable",
        "Actual Distressed"
    ],
    columns=[
        "Pred Healthy",
        "Pred Vulnerable",
        "Pred Distressed"
    ]
)

print("\n" + "=" * 80)
print("FINAL OOT CONFUSION MATRIX")
print("=" * 80)

print(cm_df)

# 10. CLASS-WISE METRICS

class_performance = pd.DataFrame({
    "Class": class_names,
    "Precision": [
        report_dict["Healthy"]["precision"],
        report_dict["Vulnerable"]["precision"],
        report_dict["Distressed"]["precision"]
    ],
    "Recall": [
        report_dict["Healthy"]["recall"],
        report_dict["Vulnerable"]["recall"],
        report_dict["Distressed"]["recall"]
    ],
    "F1": [
        report_dict["Healthy"]["f1-score"],
        report_dict["Vulnerable"]["f1-score"],
        report_dict["Distressed"]["f1-score"]
    ],
    "Support": [
        report_dict["Healthy"]["support"],
        report_dict["Vulnerable"]["support"],
        report_dict["Distressed"]["support"]
    ]
})

print("\n" + "=" * 80)
print("FINAL CLASS-WISE PERFORMANCE")
print("=" * 80)

print(class_performance.to_string(index=False))

# 11. ONE-VS-REST ROC CURVES

print("\n" + "=" * 80)
print("ROC-AUC BY CLASS")
print("=" * 80)

roc_results = []

for i, class_name in enumerate(class_names):

    y_binary = (y_oot == i).astype(int)

    class_auc = roc_auc_score(
        y_binary,
        oot_prob[:, i]
    )

    fpr, tpr, _ = roc_curve(
        y_binary,
        oot_prob[:, i]
    )

    roc_results.append({
        "Class": class_name,
        "ROC_AUC": class_auc
    })

    plt.figure(figsize=(7, 5))
    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} AUC = {class_auc:.4f}"
    )
    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"OOT ROC Curve — {class_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

roc_df = pd.DataFrame(roc_results)

print(roc_df.to_string(index=False))

# 12. OOT PREDICTED PROBABILITY SUMMARY

prob_summary = pd.DataFrame({
    "Healthy_PD": oot_prob[:, 0],
    "Vulnerable_PD": oot_prob[:, 1],
    "Distressed_PD": oot_prob[:, 2]
}).describe().T

print("\n" + "=" * 80)
print("OOT PREDICTED PROBABILITY DISTRIBUTION")
print("=" * 80)

print(prob_summary)

# 13. TOP FEATURES — FINAL MODEL

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": final_xgb.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 80)
print("FINAL XGBOOST FEATURE IMPORTANCE")
print("=" * 80)

print(
    importance_df.head(15).to_string(index=False)
)

# 14. ACTUAL VS PREDICTED CLASS DISTRIBUTION

actual_dist = pd.Series(
    y_oot
).map({
    0: "Healthy",
    1: "Vulnerable",
    2: "Distressed"
}).value_counts(normalize=True) * 100

pred_dist = pd.Series(
    oot_pred
).map({
    0: "Healthy",
    1: "Vulnerable",
    2: "Distressed"
}).value_counts(normalize=True) * 100

distribution_df = pd.DataFrame({
    "Actual_%": actual_dist,
    "Predicted_%": pred_dist
}).fillna(0)

print("\n" + "=" * 80)
print("ACTUAL VS PREDICTED OOT CLASS DISTRIBUTION")
print("=" * 80)

print(distribution_df)

# 15. FINAL PROJECT CONCLUSION

print("\n" + "=" * 80)
print("FINAL PROJECT CONCLUSION")
print("=" * 80)

print("""
1. OBJECTIVE
   The project developed a machine-learning framework to classify firms
   into Healthy, Vulnerable, and Distressed states using firm-level
   financial, cash-flow, leverage, liquidity, governance and trend
   variables.

2. TARGET INTEGRITY
   ICR was deliberately excluded from the predictive model because the
   original DistressLabel was found to be almost completely determined
   by ICR. This avoids direct target leakage and makes the model a
   genuine financial-distress prediction framework rather than an
   ICR-replication exercise.

3. TEMPORAL VALIDATION
   The model was evaluated using a chronological design:
       Training  : 2012–2019
       Validation: 2020–2021
       OOT       : 2022–2023

   Therefore, the final performance is evaluated on future-period data
   that was not used for model fitting.

4. MODEL COMPARISON
   Multinomial logistic regression provided a useful linear benchmark,
   but XGBoost substantially outperformed it on the OOT sample.

5. FINAL MODEL PERFORMANCE
   The XGBoost model demonstrates strong out-of-time discrimination and
   classification performance, particularly for identifying distressed
   firms.

6. ROBUSTNESS
   Robustness tests showed that predictive performance remains strong
   after removing the most important individual variables and entire
   groups of governance, trend and industry variables.

7. ECONOMIC INTERPRETABILITY
   SHAP analysis indicates that operating profitability, cash-flow
   generation, leverage, accrual quality and liquidity are the dominant
   drivers of model predictions. The results are economically consistent
   with conventional indicators of corporate financial health.

8. DISTRESSED-FIRM IDENTIFICATION
   The model is particularly effective at identifying the Distressed
   class, making it potentially useful as an early-warning screening
   mechanism for firms exhibiting financial deterioration.

9. LIMITATION
   The Vulnerable class is more difficult to distinguish from Healthy
   and Distressed firms. This is expected because Vulnerable represents
   an intermediate financial condition rather than an extreme state.

10. OVERALL CONCLUSION
    The evidence supports XGBoost as the preferred predictive model for
    this financial-distress classification problem. The model combines
    strong out-of-time predictive performance with robustness and
    economically interpretable feature drivers.

    The final model should therefore be presented as a financial
    distress early-warning / classification framework rather than as a
    direct reproduction of the underlying ICR-based distress label.
""")

# 16. SAVE FINAL OUTPUTS

output_dir = r"./data"

final_metrics.to_csv(
    os.path.join(
        output_dir,
        "distress_final_oot_metrics.csv"
    ),
    index=False
)

class_performance.to_csv(
    os.path.join(
        output_dir,
        "distress_final_oot_class_performance.csv"
    ),
    index=False
)

cm_df.to_csv(
    os.path.join(
        output_dir,
        "distress_final_oot_confusion_matrix.csv"
    )
)

roc_df.to_csv(
    os.path.join(
        output_dir,
        "distress_final_oot_roc_auc.csv"
    ),
    index=False
)

importance_df.to_csv(
    os.path.join(
        output_dir,
        "distress_final_feature_importance.csv"
    ),
    index=False
)

distribution_df.to_csv(
    os.path.join(
        output_dir,
        "distress_final_class_distribution.csv"
    )
)

print("\n" + "=" * 80)
print("STEP 10 COMPLETE")
print("=" * 80)

print("✓ Final XGBoost trained.")
print("✓ True 2022–2023 OOT evaluation completed.")
print("✓ Classification metrics calculated.")
print("✓ Confusion matrix generated.")
print("✓ Class-wise performance calculated.")
print("✓ ROC-AUC by class calculated.")
print("✓ Probability distribution reviewed.")
print("✓ Final feature importance calculated.")
print("✓ Actual vs predicted class distribution checked.")
print("✓ Final project conclusion generated.")

print("\nSAVED FILES:")
print(r"./data/distress_final_oot_metrics.csv")
print(r"./data/distress_final_oot_class_performance.csv")
print(r"./data/distress_final_oot_confusion_matrix.csv")
print(r"./data/distress_final_oot_roc_auc.csv")
print(r"./data/distress_final_feature_importance.csv")
print(r"./data/distress_final_class_distribution.csv")

print("\n" + "=" * 80)
print("FINANCIAL DISTRESS MODELING PIPELINE COMPLETE")
print("=" * 80)


## Model interpretation

SHAP is applied to a reproducible sample of the out-of-time observations. Global and class-specific importance are used to examine which financial characteristics are most influential in the model's predictions.

In [ ]:
# 6. CREATE OOT SHAP SAMPLE
# SHAP can be computationally expensive.
# Use up to 3,000 OOT observations while preserving reproducibility.

shap_n = min(3000, len(X_oot))

X_shap = X_oot.sample(
    n=shap_n,
    random_state=42
).copy()

y_shap = y_oot.loc[X_shap.index]

print("\nSHAP SAMPLE")
print("-" * 80)
print("Observations:", len(X_shap))

# 7. CREATE SHAP EXPLAINER

print("\n" + "=" * 80)
print("CREATING SHAP EXPLAINER")
print("=" * 80)

explainer = shap.TreeExplainer(final_xgb)

shap_values = explainer.shap_values(X_shap)

print("✓ SHAP values generated.")

# 8. HANDLE SHAP OUTPUT FORMAT

print("\nSHAP OUTPUT TYPE:", type(shap_values))

if isinstance(shap_values, list):

    # Older SHAP versions:
    # list of arrays, one array per class

    shap_class_arrays = shap_values

else:

    shap_array = np.asarray(shap_values)

    print("SHAP array shape:", shap_array.shape)

    if shap_array.ndim == 3:

        # Usually:
        # observations × features × classes

        if shap_array.shape[1] == len(feature_cols):

            shap_class_arrays = [
                shap_array[:, :, k]
                for k in range(shap_array.shape[2])
            ]

        # Alternative:
        # observations × classes × features

        elif shap_array.shape[2] == len(feature_cols):

            shap_class_arrays = [
                shap_array[:, k, :]
                for k in range(shap_array.shape[1])
            ]

        else:

            raise ValueError(
                f"Unexpected SHAP shape: {shap_array.shape}"
            )

    elif shap_array.ndim == 2:

        # Binary-style output
        shap_class_arrays = [shap_array]

    else:

        raise ValueError(
            f"Unexpected SHAP dimensions: {shap_array.ndim}"
        )

print("Number of SHAP class arrays:", len(shap_class_arrays))

# 9. CLASS NAMES

class_names = {
    0: "Healthy",
    1: "Vulnerable",
    2: "Distressed"
}

# 10. GLOBAL SHAP IMPORTANCE

print("\n" + "=" * 80)
print("GLOBAL SHAP IMPORTANCE")
print("=" * 80)

# Average absolute SHAP across all classes

global_importance_values = np.zeros(len(feature_cols))

for arr in shap_class_arrays:

    global_importance_values += np.mean(
        np.abs(arr),
        axis=0
    )

global_importance_values /= len(shap_class_arrays)

global_shap_importance = pd.DataFrame({
    "Feature": feature_cols,
    "Mean_Absolute_SHAP": global_importance_values
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

print(
    global_shap_importance.head(20).to_string(index=False)
)

# 11. CLASS-SPECIFIC SHAP IMPORTANCE

class_importance_tables = {}

for cls, class_name in class_names.items():

    if cls >= len(shap_class_arrays):
        continue

    values = np.mean(
        np.abs(shap_class_arrays[cls]),
        axis=0
    )

    table = pd.DataFrame({
        "Feature": feature_cols,
        "Mean_Absolute_SHAP": values
    }).sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    )

    class_importance_tables[class_name] = table

    print("\n" + "=" * 80)
    print(f"SHAP IMPORTANCE — {class_name.upper()}")
    print("=" * 80)

    print(
        table.head(15).to_string(index=False)
    )

# 12. SHAP SUMMARY PLOT — GLOBAL

print("\n" + "=" * 80)
print("GLOBAL SHAP SUMMARY PLOT")
print("=" * 80)

# Combine class-specific absolute effects for a global plot

global_abs_shap = np.mean(
    np.stack(
        [np.abs(arr) for arr in shap_class_arrays],
        axis=0
    ),
    axis=0
)

global_signed_shap = np.mean(
    np.stack(
        shap_class_arrays,
        axis=0
    ),
    axis=0
)

shap.summary_plot(
    global_signed_shap,
    X_shap,
    show=True,
    max_display=20
)

# 13. CLASS-SPECIFIC SUMMARY PLOTS

for cls, class_name in class_names.items():

    if cls >= len(shap_class_arrays):
        continue

    print("\n" + "=" * 80)
    print(f"SHAP SUMMARY — {class_name.upper()}")
    print("=" * 80)

    shap.summary_plot(
        shap_class_arrays[cls],
        X_shap,
        show=True,
        max_display=15
    )

# 14. TOP FEATURES FOR EACH CLASS

print("\n" + "=" * 80)
print("TOP 10 FEATURES BY CLASS")
print("=" * 80)

for class_name, table in class_importance_tables.items():

    print(f"\n{class_name}:")
    print(
        table.head(10).to_string(index=False)
    )
